In [1]:
import ConnectionConfig as cc
from pyspark.sql.functions import col, unix_timestamp, abs, md5, concat_ws, expr
from delta import DeltaTable
cc.setupEnvironment()
##comment

In [2]:
spark = cc.startLocalCluster("factRides")
spark.getActiveSession()

In [3]:
cc.set_connectionProfile("default")

rides_df = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "rides")  \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 100000) \
    .load()

rides_df.createOrReplaceTempView("rides_source")
rides_df.show(20)


+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+
|rideid|       startpoint|         endpoint|          starttime|            endtime|vehicleid|subscriptionid|startlockid|endlockid|
+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+
|     1|(51.2083,4.44595)|(51.1938,4.40228)|2015-09-22 00:00:00|2012-09-22 00:00:00|      844|         13296|       4849|     3188|
|     2|(51.2174,4.41597)|(51.2188,4.40935)|2015-09-22 00:00:00|2012-09-22 00:00:00|     4545|         45924|       NULL|     NULL|
|     3|(51.2088,4.40834)|(51.2077,4.39846)|2015-09-22 00:00:00|2012-09-22 00:00:00|     3419|         25722|       2046|     1951|
|     4|(51.2023,4.41208)|(51.2119,4.39894)|2015-09-22 00:00:00|2012-09-22 00:00:00|     1208|         31000|       1821|     2186|
|     5|(51.1888,4.45039)|(51.2221,4.40467)|2015-09-22 00:00:00|2012-09-22 0

In [4]:

subscription_df = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscriptions")  \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 100000) \
    .load()

subscription_df.createOrReplaceTempView("subscriptions_source")
subscription_df.show(20)

+--------------+----------+------------------+------+
|subscriptionid| validfrom|subscriptiontypeid|userid|
+--------------+----------+------------------+------+
|             1|2019-08-02|                 3|     1|
|             2|2019-11-12|                 1|     1|
|             3|2020-12-14|                 1|     1|
|             4|2021-10-05|                 2|     2|
|             5|2022-09-17|                 3|     3|
|             6|2019-04-08|                 1|     4|
|             7|2022-05-16|                 3|     5|
|             8|2019-06-29|                 3|     6|
|             9|2023-11-30|                 3|     6|
|            10|2023-12-31|                 3|     6|
|            11|2021-12-01|                 2|     7|
|            12|2019-02-01|                 2|     8|
|            13|2023-11-26|                 3|     8|
|            14|2021-05-08|                 3|     9|
|            15|2023-08-15|                 3|     9|
|            16|2023-11-29| 

In [5]:
locks_df = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "locks")  \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "lockid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 100000) \
    .load()

locks_df.createOrReplaceTempView("locks_source")
locks_df.show(20)

+------+-------------+---------+---------+
|lockid|stationlocknr|stationid|vehicleid|
+------+-------------+---------+---------+
|     1|            1|        1|     NULL|
|     2|            2|        1|     NULL|
|     3|            3|        1|     NULL|
|     4|            4|        1|     NULL|
|     5|            5|        1|     NULL|
|     6|            6|        1|     NULL|
|     7|            7|        1|     NULL|
|     8|            8|        1|     NULL|
|     9|            9|        1|     NULL|
|    10|           10|        1|     NULL|
|    11|           11|        1|     NULL|
|    12|           12|        1|     NULL|
|    13|           13|        1|     NULL|
|    14|           14|        1|     NULL|
|    15|           15|        1|     NULL|
|    16|           16|        1|     NULL|
|    17|           17|        1|     NULL|
|    18|           18|        1|     1664|
|    19|            1|        2|     NULL|
|    20|            2|        2|     NULL|
+------+---

In [6]:
bike_lots_df = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "bikelots")  \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "bikelotid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 100000) \
    .load()

bike_lots_df.createOrReplaceTempView("bike_lots")
bike_lots_df.show(20)

+---------+------------+----------+
|bikelotid|deliverydate|biketypeid|
+---------+------------+----------+
|        1|  2014-01-06|         1|
|        2|  2014-02-03|         1|
|        3|  2014-03-12|         1|
|        4|  2014-06-01|         1|
|        5|  2015-02-25|         1|
|        6|  2015-06-30|         1|
|        7|  2016-03-12|         1|
|        8|  2016-12-12|         1|
|        9|  2017-02-10|         3|
|       10|  2018-01-12|         2|
|       11|  2018-10-20|         3|
|       12|  2019-04-12|         4|
+---------+------------+----------+



In [7]:
vehicles_df = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "vehicles")  \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "vehicleid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 100000) \
    .load()

vehicles_df.createOrReplaceTempView("vehicles")
vehicles_df.show(20)


+---------+------------+---------+-------------------+------+-----------------+
|vehicleid|serialnumber|bikelotid|  lastmaintenanceon|lockid|         position|
+---------+------------+---------+-------------------+------+-----------------+
|        1|        1000|        1|2020-01-19 02:14:57|  NULL|(51.1968,4.40579)|
|        2|        2000|        1|2020-03-08 01:49:24|  NULL|(51.2177,4.42075)|
|        3|        3000|        1|2020-06-01 12:37:26|  1568|(51.1926,4.42151)|
|        4|        4000|        1|2020-02-27 03:13:56|  NULL|(51.2311,4.41267)|
|        5|        5000|        1|2021-03-21 03:38:31|  NULL|(51.2177,4.42075)|
|        6|        6000|        1|2020-06-16 21:44:19|  NULL|(51.2195,4.41169)|
|        7|        7000|        1|2019-10-01 10:29:21|  1556| (51.2273,4.4307)|
|        8|        8000|        1|2019-12-06 17:09:49|  NULL|(51.2047,4.39625)|
|        9|        9000|        1|2020-01-01 10:06:51|  NULL|(51.2058,4.41837)|
|       10|       10000|        1|2019-1

In [8]:
stations_df = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations")  \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "stationid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 100000) \
    .load()

stations_df.createOrReplaceTempView("stations")
stations_df.show(20)

+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|stationid|objectid|stationnr|        type|              street| number|zipcode|  district|         gpscoord|      additionalinfo|labelid|cityid|
+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|        1|   33202|      026|DUBBELZIJDIG|         Meir (2000)|     84|   2000| ANTWERPEN|(51.2182,4.41241)|                    |   NULL|  NULL|
|        2|   33203|      019| ENKELZIJDIG|          ONTBREKEND|     12|   2000| ANTWERPEN| (51.219,4.40405)|                    |   NULL|  NULL|
|        3|   33204|      020| ENKELZIJDIG|Groenkerkhofstraa...|      2|   2000| ANTWERPEN|(51.2187,4.40066)| thv Nationalestraat|   NULL|  NULL|
|        4|   33205|      035| ENKELZIJDIG|Cockerillkaai (2000)|       |   2000| ANTWERPEN|(51.2104,4.38772)|               

In [9]:
dim_date = spark.read.format("delta").load("spark-warehouse/dimdate")
dim_date.createOrReplaceTempView("dimDate")

dim_user = spark.read.format("delta").load("spark-warehouse/userdim")
dim_user.createOrReplaceTempView("dimUser")

dim_weather = spark.read.format("delta").load("spark-warehouse/weatherdim")
dim_weather.createOrReplaceTempView("dimWeather")


dim_station = spark.read.format("delta").load("spark-warehouse/stationdim")
dim_station.createOrReplaceTempView("dimStation")
#dim_vehicle_type = spark.read.format("delta").load("spark-warehouse/vehicletypedim")





In [10]:
# Define the Haversine formula
from pyspark.sql.functions import udf
from math import radians, sin, cos, sqrt, atan2
def haversine_km(lat1, lon1, lat2, lon2):
    # Radius of Earth in kilometers
    R = 6371.0

    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    # Differences in coordinates
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Haversine formula
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Register as a Spark UDF
haversine_udf = udf(haversine_km)
spark.udf.register("haversine_km", haversine_udf)


In [11]:
from pyspark.sql.functions import col, unix_timestamp, abs, md5, concat_ws
# Load weather JSON files into a DataFrame
weather_df = spark.read.option("multiline", "true").json("weatherdata/")
weather_df = weather_df.withColumn("weather_timestamp", unix_timestamp(col("timestamp")))

# Create temp views
weather_df.createOrReplaceTempView("weather_source")


In [12]:
from pyspark.sql.functions import regexp_extract, when, col, lit, monotonically_increasing_id, input_file_name

# Load JSONs
weather_df = spark.read.option("multiline", "true").json("weatherdata/*.json")

# Extract zip code from filename
weather_df = weather_df.withColumn(
    "zip_code", regexp_extract(input_file_name(), r"([^/]+)\.json$", 1)
)

from pyspark.sql.functions import when, col, lit

weather_df.select("precipitation", "temperature_2m").show(5)
weather_df.select("precipitation", "temperature_2m").printSchema()


# Ensure columns are treated as numerical (e.g., float)
weather_df = weather_df.withColumn("precipitation", col("precipitation").cast("double"))
weather_df = weather_df.withColumn("temperature_2m", col("temperature_2m").cast("double"))

from pyspark.sql.functions import expr

weather_df = weather_df.withColumn(
    "weather_type",
    expr("""
        CASE
            WHEN precipitation > 0 THEN 'Unpleasant'
            WHEN precipitation = 0 AND temperature_2m > 15 THEN 'Pleasant'
            WHEN precipitation = 0 AND temperature_2m <= 15 THEN 'Neutral'
            ELSE 'Unknown'
        END
    """)
)


weather_df.show(100)


+-------------+------------------+
|precipitation|    temperature_2m|
+-------------+------------------+
|          0.0|17.213001251220703|
|          0.0| 16.51300048828125|
|          0.0|16.113000869750977|
|          0.0|17.163000106811523|
|          0.0| 20.26300048828125|
+-------------+------------------+
only showing top 5 rows

root
 |-- precipitation: double (nullable = true)
 |-- temperature_2m: double (nullable = true)

+---------+---------+-------------------+------------------+--------------------+--------+------------+
| latitude|longitude|      precipitation|    temperature_2m|           timestamp|zip_code|weather_type|
+---------+---------+-------------------+------------------+--------------------+--------+------------+
|51.219464| 4.397171|                0.0|17.213001251220703|2023-08-15T00:00:...|    2000|    Pleasant|
|51.219464| 4.397171|                0.0| 16.51300048828125|2023-08-15T02:00:...|    2000|    Pleasant|
|51.219464| 4.397171|                0.0|16

In [13]:
from pyspark.sql.functions import (
    unix_timestamp, to_date, col, abs as abs_diff, row_number, md5, concat_ws, coalesce
)
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Create dimWeather
#weather_df.createOrReplaceTempView("dimWeather")
#weather_df.show(4)

# Continue with rides fact creation
rides_df.createOrReplaceTempView("rides_source")
from pyspark.sql.functions import to_timestamp, unix_timestamp

# Parse ISO 8601 timestamp in weather data
weather_df = weather_df.withColumn("timestamp", to_timestamp("timestamp"))

# Optionally create a unix timestamp column for join logic in weather data
weather_df = weather_df.withColumn("timestamp_unix", unix_timestamp("timestamp"))

weather_df.createOrReplaceTempView("weatherdata")

# Modify the query to convert ride_starttime to Unix timestamp if not already done
ridesFactFromSource = spark.sql("""
WITH base AS (
    SELECT 
        r.rideid,
        usrdim.user_sk AS user_sk,
        locks.stationid AS start_station_id,
        endlocks.stationid AS end_station_id,
        dimDate.dateSK AS date_sk,
        bike_lots.biketypeid AS vehicle_id,
        unix_timestamp(r.starttime) AS ride_starttime,
        REGEXP_EXTRACT(CAST(station.zipcode AS STRING), '\\\\d{4}', 0) AS start_zip_code
    FROM rides_source r
    JOIN subscriptions_source sub ON sub.subscriptionid = r.subscriptionid
    JOIN dimUser usrdim ON usrdim.userid = sub.userid
    JOIN locks_source locks ON r.startlockid = locks.lockid
    JOIN locks_source endlocks ON r.endlockid = endlocks.lockid
    JOIN dimStation station ON station.stationid = locks.stationid
    JOIN dimDate ON to_date(r.starttime) = dimDate.calendarDate
    JOIN vehicles ON vehicles.vehicleid = r.vehicleid
    JOIN bike_lots ON bike_lots.bikelotid = vehicles.bikelotid
    WHERE usrdim.is_current = true
),
weather_match AS (
    SELECT 
        b.*,
        w.precipitation,
        w.weather_type,
        ROW_NUMBER() OVER (
            PARTITION BY b.rideid 
            ORDER BY ABS(b.ride_starttime - w.timestamp_unix)
        ) AS rn
    FROM base b
    LEFT JOIN weatherdata w 
      ON w.zip_code = b.start_zip_code
),
final_match AS (
    SELECT wm.*, dw.weather_id
    FROM weather_match wm
    LEFT JOIN dimWeather dw ON wm.weather_type = dw.weather_type
    WHERE rn = 1
)
SELECT
    rideid, 
    user_sk, 
    start_station_id, 
    end_station_id, 
    date_sk, 
    vehicle_id, 
    COALESCE(precipitation, 0) AS precipitation,
    CAST(COALESCE(weather_id, -1) AS INT) AS weather_sk,
    md5(concat_ws('||', 
        rideid, 
        user_sk, 
        start_station_id, 
        end_station_id, 
        date_sk, 
        vehicle_id, 
        COALESCE(precipitation, 0),
        COALESCE(weather_id, -1)
    )) AS source_md5
FROM final_match
""")

ridesFactFromSource.groupBy("weather_sk").count().orderBy("weather_sk").show()


+----------+-------+
|weather_sk|  count|
+----------+-------+
|         1|  10874|
|         2|3586204|
|         3|   6270|
+----------+-------+



In [ ]:
ridesFactFromSource.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("factRides")


In [15]:

ridesFactFromSource.repartition(1).write.format("parquet").mode("overwrite").saveAsTable("FactRides_pq")

ridesFactFromSource.groupBy("weather_sk").count().orderBy("weather_sk").show()

SparkRuntimeException: [LOCATION_ALREADY_EXISTS] Cannot name the managed table as `spark_catalog`.`default`.`factrides_pq`, as its associated location 'file:/C:/kdg/data4/shared/sharedbicycleproject-team46/sharedBicycleProject_team46/DimTables/spark-warehouse/spark-warehouse/factrides_pq' already exists. Please pick a different table name, or remove the existing location first.

In [ ]:
abc = spark.read.parquet("./spark-warehouse/factrides_pq/part-00000-cf31336a-8c28-469c-9a22-44ac0727486e-c000.snappy.parquet")

In [20]:
abc.groupBy("weather_sk").count().orderBy("weather_sk").show()

+----------+-------+
|weather_sk|  count|
+----------+-------+
|         1|  36332|
|         3|3567016|
+----------+-------+



In [16]:
ridesFactFromSource.createOrReplaceTempView("ridesFactNew")

In [17]:
dt_ridesFact = DeltaTable.forPath(spark,"./spark-warehouse/factrides")
dt_ridesFact.toDF().createOrReplaceTempView("ridesFactCurrent")

result = spark.sql("MERGE INTO ridesFactCurrent AS target \
      using ridesFactNew AS source ON target.rideid = source.rideid \
      WHEN MATCHED and source.source_md5<>target.source_md5 THEN UPDATE SET * \
      WHEN NOT MATCHED THEN INSERT *")
result.show(100)

AnalysisException: [DELTA_MISSING_DELTA_TABLE] `./spark-warehouse/factrides` is not a Delta table.

In [15]:
#THE THINGS LEFT TO DO:



#RECEIVE DATA FROM WEATHER API
#DEPENDING ON THE TEMPERATURE AND WEATHER CONDITION SELECT THE APPROPRIATE WEATHER_TYPE
#THEN ADD THIS WEATHER_ID TO THE THIS FACT TABLE

In [16]:
spark.stop()

In [17]:

# Run the SQL query
query = """
SELECT 
    b.rideid, 
    b.ride_starttime,
    w.timestamp AS weather_timestamp,
    w.timestamp_unix AS weather_timestamp_unix,
    ABS(b.ride_starttime - w.timestamp_unix) AS timestamp_diff,
    ROW_NUMBER() OVER (
        PARTITION BY b.rideid 
        ORDER BY ABS(b.ride_starttime - w.timestamp_unix)
    ) AS rn
FROM base b
LEFT JOIN weatherdata w 
  ON w.zip_code = b.start_zip_code
LIMIT 10;
"""

# Execute the query and show the results
result = spark.sql(query)
result.show(10)  # Display the first 10 rows of the result



AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `base` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 12 pos 5;
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Project ['b.rideid, 'b.ride_starttime, 'w.timestamp AS weather_timestamp#6286, 'w.timestamp_unix AS weather_timestamp_unix#6287, 'ABS(('b.ride_starttime - 'w.timestamp_unix)) AS timestamp_diff#6288, row_number() windowspecdefinition('b.rideid, 'ABS(('b.ride_starttime - 'w.timestamp_unix)) ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rn#6289]
      +- 'Join LeftOuter, ('w.zip_code = 'b.start_zip_code)
         :- 'SubqueryAlias b
         :  +- 'UnresolvedRelation [base], [], false
         +- SubqueryAlias w
            +- 